# Lab 2: Search and A*
Source: [search_lab_ex.pdf](search_lab_ex.pdf). Tasks follow the sheet's order.

LLM assistance: Codex prepared the code and concise written answers; saved outputs record executed checks.


## Task 0: Understand the search problem

| Component | Specification |
|---|---|
| State S | A free-cell position `(row, column)` |
| Actions A | Up, down, left, right |
| Transition T | Add the move's displacement if the destination is free |
| Initial state | `S`, at `(1, 1)` |
| Goal G | `G`, at `(7, 15)` |
| Cost c | One per move |

(a) Row and column specify a state because the map is fixed.
(b) A move is invalid if it leaves the map or enters `#`.
(c) The problem is deterministic: each valid move has one successor.
(d) A solution is a legal sequence from `S` to `G`; an optimal solution has the fewest moves.

**Think about it:** defining states, legal transitions, and costs first gives concrete criteria for checking the generated program.


## Task 1: Plan the agent
Use tuples for positions and strings for map rows. Generate valid neighboring positions after bounds and obstacle checks.
A min-heap stores `(g + h, insertion_order, state, g)`; a dictionary stores the cheapest known cost to each state.
Stop when the popped state is `G`. Follow parent links backward, then reverse them.
Report success, path, move count, and expanded states (a state is counted when its successors are generated; the goal is excluded).

**Think about it:** this design fixes what must be implemented and measured before code generation.


## Task 2: Generate A*
**Implementation prompt used for this LLM-assisted submission:**

> Implement A* in readable Python for the exact warehouse map below. Use (row, column) states, four legal unit-cost moves, Manhattan distance, a heap frontier with FIFO tie-breaking, cheapest-known costs, stale-entry checks, and parent links. Reopen a state if a cheaper path is found. Report success, the complete path, move count, and states expanded, excluding the goal. Keep the heuristic configurable for later experiments. Do not change the map.


In [1]:
from collections import deque
from heapq import heappop, heappush
from itertools import count
from math import hypot

warehouse = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]


def locate(grid, symbol):
    positions = [(r, c) for r, row in enumerate(grid)
                 for c, value in enumerate(row) if value == symbol]
    if len(positions) != 1:
        raise ValueError(f"Expected exactly one {symbol}")
    return positions[0]


def neighbors(grid, state):
    row, column = state
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        r, c = row + dr, column + dc
        if 0 <= r < len(grid) and 0 <= c < len(grid[r]) and grid[r][c] != "#":
            yield r, c


def manhattan(state, goal):
    return abs(state[0] - goal[0]) + abs(state[1] - goal[1])


def reconstruct(parent, state):
    path = []
    while state is not None:
        path.append(state)
        state = parent[state]
    return path[::-1]


def astar(grid, heuristic=manhattan):
    start, goal = locate(grid, "S"), locate(grid, "G")
    order = count()
    frontier = [(heuristic(start, goal), next(order), start, 0)]
    costs, parent = {start: 0}, {start: None}
    expanded = 0
    while frontier:
        _, _, state, g = heappop(frontier)
        if g != costs[state]:
            continue  # A cheaper route superseded this entry.
        if state == goal:
            return reconstruct(parent, state), expanded
        expanded += 1
        for successor in neighbors(grid, state):
            new_g = g + 1
            if new_g < costs.get(successor, float("inf")):
                costs[successor] = new_g
                parent[successor] = state
                f = new_g + heuristic(successor, goal)
                heappush(frontier, (f, next(order), successor, new_g))
    return None, expanded


def report(name, result):
    path, expanded = result
    print(name)
    print("Solution found:", path is not None)
    print("Path:", path if path is not None else "No path found")
    print("Path length:", len(path) - 1 if path is not None else None)
    print("States expanded:", expanded)


original_result = astar(warehouse)
report("Original warehouse", original_result)


Original warehouse
Solution found: True
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Path length: 40
States expanded: 63


## Task 3: Test the generated program
Check every returned move and compare its length with an independent BFS implementation.
The alternative-path map has a known two-step route and a longer route through the row below it.


In [2]:
def bfs(grid):
    start, goal = locate(grid, "S"), locate(grid, "G")
    frontier, parent = deque([start]), {start: None}
    expanded = 0
    while frontier:
        state = frontier.popleft()
        if state == goal:
            return reconstruct(parent, state), expanded
        expanded += 1
        for successor in neighbors(grid, state):
            if successor not in parent:
                parent[successor] = state
                frontier.append(successor)
    return None, expanded


def valid_path(grid, path):
    if path is None:
        return False
    return (path[0] == locate(grid, "S") and path[-1] == locate(grid, "G")
            and all(grid[r][c] != "#" for r, c in path)
            and all(b in list(neighbors(grid, a)) for a, b in zip(path, path[1:])))


test_maps = {
    "1. Original": warehouse,
    "2. Adjacent goal": ["#####", "#SG##", "#####"],
    "3. No solution": ["#######", "#S....#", "###.###", "#...#G#", "#######"],
    "4. Alternative paths": ["#######", "#S.G..#", "#.....#", "#######"],
}
for name, grid in test_maps.items():
    result = astar(grid)
    reference, _ = bfs(grid)
    path, _ = result
    if reference is None:
        assert path is None
    else:
        assert valid_path(grid, path) and len(path) == len(reference)
    report(name, result)
    print()
assert len(astar(test_maps["2. Adjacent goal"])[0]) - 1 == 1
assert len(astar(test_maps["4. Alternative paths"])[0]) - 1 == 2
print("All four tests passed")


1. Original
Solution found: True
Path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
Path length: 40
States expanded: 63

2. Adjacent goal
Solution found: True
Path: [(1, 1), (1, 2)]
Path length: 1
States expanded: 1

3. No solution
Solution found: False
Path: No path found
Path length: None
States expanded: 9

4. Alternative paths
Solution found: True
Path: [(1, 1), (1, 2), (1, 3)]
Path length: 2
States expanded: 2

All four tests passed


**Think about it:** known one-step, unreachable, and shortest-path cases check properties that a plausible-looking route cannot establish.

## Task 4: Inspect A*

| Concept | Code |
|---|---|
| State | `state` tuple |
| Action / transition | Displacements and checks in `neighbors` |
| Goal test | `state == goal` |
| g(n) | `g`, stored in `costs` |
| h(n) | `heuristic(successor, goal)` |
| f(n) | `new_g + heuristic(successor, goal)` |
| Frontier | `frontier`, managed with `heappush` / `heappop` |
| Discovered states | `costs`; only strictly cheaper routes are enqueued |
| Path reconstruction | `parent` and `reconstruct` |

(a) A min-heap. (b) Pop the smallest `f`, breaking ties by insertion order.
(c) `manhattan` computes the default heuristic. (d) `f` is explicitly computed.
(e) Cost checks prevent redundant routes, and stale heap entries are skipped; a cheaper route may reopen a state.


## Task 5: Compare A* with blind search
**Follow-up implementation prompt:**

> Implement BFS for the unchanged warehouse and use the same neighbor order, parent reconstruction, and expansion counting as A*. Compare success, path length, and expanded states.


In [3]:
comparison = {"BFS": bfs(warehouse), "A*": astar(warehouse)}
print(f"{'Algorithm':<12} {'Found':<8} {'Moves':<8} {'Expanded':<8}")
for name, (path, expanded) in comparison.items():
    print(f"{name:<12} {str(path is not None):<8} {len(path)-1:<8} {expanded:<8}")
bfs_path, bfs_expanded = comparison["BFS"]
astar_path, astar_expanded = comparison["A*"]
assert len(bfs_path) == len(astar_path)
print("Both found a solution of the same length.")
if astar_expanded < bfs_expanded:
    print("A* expanded fewer states.")
elif astar_expanded == bfs_expanded:
    print("Both expanded the same number of states.")
else:
    print("BFS expanded fewer states.")


Algorithm    Found    Moves    Expanded
BFS          True     40       63      
A*           True     40       63      
Both found a solution of the same length.
Both expanded the same number of states.


A* can expand fewer states because `h` favors positions nearer the goal. Obstacles and ties can weaken that advantage; the table measures this map only.
**Think about it:** BFS uses path depth; A* also uses an estimate of remaining cost.

## Task 6: Investigate the heuristic
**Explanation prompt:**

> Explain why Manhattan distance is a lower bound on a four-direction unit-cost warehouse route, and distinguish admissible estimates from twice Manhattan distance. Check the claims using the unchanged-map experiments below.

Manhattan counts the required horizontal and vertical displacement while ignoring obstacles, so it never overestimates the shortest route. It is also consistent across adjacent cells.


In [4]:
heuristics = {
    "Manhattan": manhattan,
    "Zero": lambda state, goal: 0,
    "Euclidean": lambda state, goal: hypot(state[0]-goal[0], state[1]-goal[1]),
    "2 x Manhattan": lambda state, goal: 2 * manhattan(state, goal),
}
print(f"{'Heuristic':<18} {'Found':<8} {'Moves':<8} {'Expanded':<8}")
for name, heuristic in heuristics.items():
    path, expanded = astar(warehouse, heuristic)
    assert valid_path(warehouse, path)
    if name != "2 x Manhattan":
        assert len(path) == len(bfs_path)
    print(f"{name:<18} {str(path is not None):<8} {len(path)-1:<8} {expanded:<8}")


Heuristic          Found    Moves    Expanded
Manhattan          True     40       63      
Zero               True     40       63      
Euclidean          True     40       63      
2 x Manhattan      True     40       66      


`h = 0` gives uniform-cost search, which has BFS ordering here. Euclidean is admissible but no larger than Manhattan, so it provides a weaker lower bound. Doubling Manhattan can overestimate and removes the optimality guarantee, even if this map still produces a shortest path.

**Think about it:** a weak optimistic estimate may increase exploration; an aggressive overestimate may reduce it while sacrificing shortest-path guarantees. One successful map cannot prove admissibility.

## Task 7: Evaluate the LLM-generated agent
1. The submitted heap, cost updates, bounds checks, and parent links passed execution tests.
2. No implementation failure was observed in these tests; tie-breaking and expansion counting needed explicit definitions.
3. Known outcomes and independent BFS checked failure handling, route validity, and shortest lengths.
4. Heap priorities, FIFO ties, and stale entries are identified above.
5. BFS and configurable heuristics implement the sheet's requested extensions; the warehouse is unchanged.
6. The unreachable map checks termination; alternative routes plus BFS check optimality.
7. Generated code needs execution and independent checks before acceptance.
8. A* combines accumulated cost with an estimate; an estimate can guide search without guaranteeing improvement on every map.

All design text and code here were LLM-assisted; executed checks provide evidence separately from the explanations.

## Final reflection
1. A precise formulation defines valid states, moves, and costs, making implementation errors detectable.
2. A* is informed because its priority includes an estimate of remaining cost.
3. Heuristic strength affects expansion count; overestimation can lose optimality.
4. Codex translated the design into code, tests, and compact explanations.
5. Unchecked code could cross obstacles, loop on failure, miscount moves, or return a longer route while claiming it is shortest.
